# Exercice 5 ★★★★☆ — Hiérarchie complète

**Comment faire :**
1. Lancez la cellule des imports.
2. Écrivez votre code dans la cellule « Votre code », sous le `TODO`, puis lancez-la.
3. Lancez la cellule « Vérification » et lisez les messages ❌ : ils disent ce qui manque.
4. Corrigez et relancez autant de fois que nécessaire.

Bloqué·e ? Dépliez l'indice. En dernier recours : `correction_exercice5.ipynb`.

## Énoncé

En suivant le motif `HeroBase / Hero / HeroCreate / HeroPublic` de la présentation, construisez la hiérarchie pour une entité `Book` :

- `BookBase` : champs communs `title: str` et `author: str`
- `Book` : la vraie table — ajoute `id` (clé primaire) et `isbn: str`
- `BookCreate` : schéma d'entrée API (mêmes champs que `BookBase`, plus `isbn`)
- `BookPublic` : schéma de sortie API (mêmes champs que `BookBase`, plus `id`)

In [ ]:
from sqlmodel import SQLModel, Field, Relationship
from sqlalchemy.orm import clear_mappers

## Votre code

In [ ]:
# Ne pas toucher : permet de relancer cette cellule sans erreur.
SQLModel.metadata.clear()
clear_mappers()

# TODO — Exercice 5 : créez BookBase (title, author), Book (table : + id, isbn),
#                     BookCreate (+ isbn) et BookPublic (+ id).



<details>
<summary>💡 Indice</summary>

Quatre classes, une seule avec `table=True`. Les trois autres héritent de `BookBase` sans réécrire `title` ni `author`.
</details>

## Vérification

In [ ]:
import pytest

def get_fields(model):
    """Compatible pydantic v1 (SQLModel ancien) et v2 (SQLModel récent)."""
    if hasattr(model, "model_fields"):
        return model.model_fields
    return model.__fields__


def is_table(model) -> bool:
    return getattr(model, "__table__", None) is not None


# ---------------------------------------------------------------------------
# Exercice 5 — hiérarchie Book
# ---------------------------------------------------------------------------

def test_exercice5_classes_existent():
    for nom in ("BookBase", "Book", "BookCreate", "BookPublic"):
        assert hasattr(ex, nom), f"La classe {nom} est introuvable."


def test_exercice5_bookbase_pas_une_table():
    assert not is_table(ex.BookBase), "BookBase ne doit PAS être une table."


def test_exercice5_bookbase_champs_communs():
    fields = get_fields(ex.BookBase)
    for champ in ("title", "author"):
        assert champ in fields, f"Le champ '{champ}' est manquant sur BookBase."


def test_exercice5_book_est_une_table():
    assert is_table(ex.Book), "Book doit être déclarée avec table=True."


def test_exercice5_book_a_tous_les_champs():
    fields = get_fields(ex.Book)
    for champ in ("title", "author", "id", "isbn"):
        assert champ in fields, f"Le champ '{champ}' est manquant sur Book."


def test_exercice5_bookcreate_bookpublic():
    fields_create = get_fields(ex.BookCreate)
    for champ in ("title", "author", "isbn"):
        assert champ in fields_create, f"Le champ '{champ}' est manquant sur BookCreate."
    fields_public = get_fields(ex.BookPublic)
    for champ in ("title", "author", "id"):
        assert champ in fields_public, f"Le champ '{champ}' est manquant sur BookPublic."


def test_exercice5_book_instanciation():
    b = ex.Book(title="Fondation", author="Isaac Asimov", isbn="978-2070368228")
    assert b.title == "Fondation"
    assert b.isbn == "978-2070368228"

# --- Lancement des tests ----------------------------------------------------
import types

ex = types.SimpleNamespace(**globals())
_ok = 0
_tests = [(nom, f) for nom, f in list(globals().items()) if nom.startswith("test_")]
for _nom, _test in _tests:
    try:
        _test()
        print("✅", _nom)
        _ok += 1
    except Exception as erreur:
        print("❌", _nom, "→", str(erreur) or type(erreur).__name__)
print(f"\n{_ok}/{len(_tests)} tests réussis" + (" — bravo !" if _ok == len(_tests) else ""))
